In [1]:
!pip install ipython matplotlib mne numpy pandas scikit_learn seaborn sktime numba mlflow optuna

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import os
from IPython.display import display
%matplotlib inline

import mne
from mne.datasets import eegbci, sample, erp_core
from mne.io import read_raw_edf, concatenate_raws
from mne.time_frequency import psd_array_welch
from mne.decoding import CSP

from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn import metrics
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.ensemble import RandomForestClassifier
from sktime.transformations.panel.rocket import Rocket
from sktime.classification.kernel_based import RocketClassifier

import mlflow
import mlflow.sklearn

import optuna

E:\develope\python\eegia\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Частота для ресемплинга
sfreq = 160
# Количество испытуемых (для датасета bci)
n_sub = 5

le = LabelEncoder()

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("EEG_Classification")

E:\develope\python\eegia\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///E:/develope/python/eegia/mlruns/961817029434495920', creation_time=1773489193483, experiment_id='961817029434495920', last_update_time=1773489193483, lifecycle_stage='active', name='EEG_Classification', tags={}, workspace='default'>

### Конвертер данных для sktime
Преобразует 3D массив (эпохи, каналы, время) в формат pd.DataFrame для sktime

In [4]:
class SktimeConverter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        n_epochs, n_channels, n_times = X.shape
        cols = [f"ch{i}" for i in range(n_channels)]
        return pd.DataFrame({
            cols[i]: [pd.Series(X[j, i, :]) for j in range(n_epochs)]
            for i in range(n_channels)
        })

### Функция визуализации матрицы ошибок

In [5]:
def show_confusion_matrix(y_encoded, y_pred, display_labels, name, dataset_id):
    plt.figure(figsize=(6, 4))
    sns.heatmap(metrics.confusion_matrix(y_encoded, y_pred), annot=True, fmt='d', cmap='Blues',
                yticklabels=display_labels)
    plt.title(f"Матрица ошибок | Модель: {name} | Датасет: {dataset_id}")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

### Гибридный экстрактор
Объединяет спектральную мощность (частотный домен) и признаки ROCKET (временной домен)

In [6]:
class EEGHybridExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, num_kernels=200, sfreq=sfreq):
        self.num_kernels = num_kernels
        self.sfreq = sfreq
        self.rocket = Rocket(num_kernels=self.num_kernels, random_state=42)
        self.scaler = StandardScaler()
        self.converter = SktimeConverter()

    def _get_band_power(self, X):
        # Расчет мощности классических ритмов ЭЭГ через метод Велча
        nyquist = self.sfreq / 2
        f_limit = min(40, nyquist - 1)
        
        n_fft = min(256, X.shape[-1])
        psds, freqs = psd_array_welch(
            X, sfreq=self.sfreq, fmin=1, fmax=f_limit, 
            n_fft=n_fft, n_overlap=n_fft//2, verbose=False
        )
        
        bands = {"delta": (1, 4), "theta": (4, 7), "alpha": (7, 14), 
                 "beta": (14, 30), "gamma": (30, f_limit)}
        
        out = []
        for band, (fmin, fmax) in bands.items():
            idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            power = psds[:, :, idx].mean(axis=-1) 
            power = 10 * np.log10(psds[:, :, idx].mean(axis=-1) + 1e-20)
            out.append(power)
        
        return np.concatenate(out, axis=1)
    def _combine_features(self, X):
        X_df = self.converter.transform(X)
        X_rocket = self.rocket.transform(X_df)
        X_bp = self._get_band_power(X)
        return np.concatenate([X_bp, np.asarray(X_rocket)], axis=1)
        
    def fit(self, X, y=None):
        X_df = self.converter.transform(X)
        self.rocket.fit(X_df)
        X_combined = self._combine_features(X)
        self.scaler.fit(X_combined)
        return self
        
    def transform(self, X):
        X_combined = self._combine_features(X)
        return self.scaler.transform(X_combined)

## Загрузка ДАТАСЕТА

Подгружаем сырые данные (Raw). Разные датасеты имеют свою специфику: EEG BCI фокусируется на моторном воображении (представлении движений), MNE Sample — на первичных сенсорных ответах (слух/зрение), а ERP Core — на когнитивном контроле (задача Фланкера). Переименование каналов и установка стандартного монтажа (10-20) необходимы для корректной пространственной локализации электродов

In [7]:
def load_selected_dataset(choice):
    match choice:
        case 1:
            print("Загрузка EEG BCI")
            subjects = range(1, n_sub + 1)
            runs = [3, 7, 11]
            all_raws = []
            for s in subjects:
                fnames = eegbci.load_data(s, runs)
                all_raws.extend([mne.io.read_raw_edf(f, preload=True) for f in fnames])
            raw = mne.concatenate_raws(all_raws)
            raw.rename_channels(lambda x: x.strip('.').replace('Z', 'z').replace('FP', 'Fp'))
            raw.set_montage('standard_1020', on_missing='ignore')
        case 2:
            print("Загрузка MNE Sample")
            data_path = sample.data_path()
            raw_fname = data_path / 'MEG' / 'sample' / 'sample_audvis_raw.fif'
            raw = mne.io.read_raw_fif(raw_fname, preload=True)
            raw.pick(['eeg', 'stim'])
            
        case 3:
            print("Загрузка ERP Core")
            raw_fname = erp_core.data_path() / f'ERP-CORE_Subject-001_Task-Flankers_eeg.fif'
            raw = mne.io.read_raw_fif(raw_fname, preload=True)
            raw.pick(['eeg', 'stim'])
        case _:
            raise ValueError("Не существует")
    
    return raw

## Функция ICA
Разбиваем многоканальный сигнал на статистически независимые источники и удаляем те, что похожи на моргание глаз. Глазные яблоки — это диполи. При моргании они создают мощный электрический импульс, который "забивает" сигналы мозга. ICA (Independent Component Analysis) позволяет отделить сигнал "моргания" от "мыслей", не удаляя при этом полезные данные из самих каналов.

In [8]:
def apply_ica(raw, n_components=20, random_state=42, threshold=2.5):
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
    n_channels = len(raw_for_ica.ch_names)
    n_components = min(n_components, n_channels - 1)
    ica = mne.preprocessing.ICA(
        n_components=n_components, 
        random_state=random_state, 
        method='fastica',
        max_iter='auto'
    )
    ica.fit(raw_for_ica, verbose=False)
    # Список каналов с потенциальным морганием
    candidate_eog = ['Fp1', 'Fp2', 'FP1', 'FP2', 'EEG 001']
    # Находим существующие в датасете
    eog_ch = [ch for ch in candidate_eog if ch in raw_for_ica.ch_names]
    if eog_ch:
        # Используем первый найденный лобный канал как референс для поиска морганий
        eog_indices, eog_scores = ica.find_bads_eog(
            raw_for_ica, 
            ch_name=eog_ch[0], 
            threshold=threshold,
            verbose=False
        )
        ica.exclude = eog_indices
    else:
        ica.exclude = []
    raw_clean = ica.apply(raw.copy(), verbose=False)
    return raw_clean

## Функция разделения на эпохи
Нарезаем непрерывную запись на короткие эпохи вокруг событий. В ЭЭГ нас интересует реакция на конкретное событие. Мы берем окно (например, от -0.2 до 0.8 сек), где 0 — момент стимула. Baseline correction вычитает среднее значение "пред-стимульного" периода, чтобы скомпенсировать возможный сдвиг напряжения в момент записи.

In [9]:
def epochs_from_raw(raw, tmin=-0.2, tmax=0.8, event_id=None, picks='data', events=None):
    if events is None:
        events, _ = mne.events_from_annotations(raw, verbose=False)
        if len(events) == 0:
            events = mne.find_events(raw, stim_channel='STI 014', verbose=False)

    unique, counts = np.unique(events[:, -1], return_counts=True)
    valid_classes = unique[counts > 5]
    events = events[np.isin(events[:, -1], valid_classes)]
    
    raw_proc = raw.copy()
    raw_proc.filter(1, 40, fir_design='firwin', picks='data', verbose=False)
    if raw_proc.info['sfreq'] > sfreq:
        raw_proc.resample(sfreq, verbose=False)
    raw_proc.set_eeg_reference('average', projection=False, verbose=False)

    raw_clean = apply_ica(raw_proc)
    
    epochs = mne.Epochs(
        raw_clean, 
        events=events, 
        event_id=event_id,
        tmin=tmin, 
        tmax=tmax,
        baseline=(None, 0),
        preload=True,
        reject=dict(eeg=150e-6),
        flat=dict(eeg=1e-6),
        verbose=False
    )

    X = epochs.get_data(picks=picks) * 1e6
    y = epochs.events[:, -1]

    print(f"Всего эпох после фильтрации: {len(epochs)}")
    print(f"Классы: {dict(zip(*np.unique(y, return_counts=True)))}")
    
    return X, y

## Подготовка размеченных данных
Сырые данные содержат множество технических меток (триггеров), не все из которых релевантны задаче классификации.
- Для MNE Sample мы объединяем разные типы звуковых и визуальных стимулов в два чистых макро-класса (Audio/Visual).
- Для ERP Core мы выделяем только те условия, которые отражают когнитивную нагрузку (Easy/Hard), отсеивая артефакты нажатия кнопок.

In [10]:
def prepare_labeled_data(raw, dataset_id):
    if dataset_id == 2:
        events = mne.find_events(raw, stim_channel='STI 014', verbose=False)
    else:
        events, _ = mne.events_from_annotations(raw, verbose=False)
        
    # Оставляем только нужные триггеры, игнорируя технические события
    if dataset_id == 2:
        mask = np.isin(events[:, -1], [1, 2, 3, 4])
        events = events[mask]
        events[np.isin(events[:, -1], [1, 2]), -1] = 101
        events[np.isin(events[:, -1], [3, 4]), -1] = 102
        
    elif dataset_id == 3:
        mask = np.isin(events[:, -1], [3, 4, 5, 6])
        events = events[mask]
        events[np.isin(events[:, -1], [3, 4]), -1] = 201
        events[np.isin(events[:, -1], [5, 6]), -1] = 202
    
    return epochs_from_raw(raw, events=events)

## Кросс-валидация и обучение
Для оценки моделей мы используем Stratified K-Fold Cross-Validation. Стратификация гарантирует, что в каждом фолде сохраняется исходное соотношение классов (например, 50% Audio и 50% Visual), что предотвращает смещение модели в сторону более частого класса.

In [11]:
def start_training(dataset_list, dataset_res, cv):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_artifact_path = os.path.join("temp_plots", timestamp)
    os.makedirs(base_artifact_path, exist_ok=True)
    
    for dataset_id in dataset_list:
        with mlflow.start_run(run_name=f"Dataset_{dataset_id}", nested=True):
            print(f"\n--- Обработка Датасета {dataset_id} ---")
            raw = load_selected_dataset(dataset_id)
            X, y = prepare_labeled_data(raw, dataset_id)
            
            y_encoded = le.fit_transform(y)
            unique_labels = le.classes_.astype(str)
            label_mapping = {'101': 'Audio', '102': 'Visual', '201': 'Easy', '202': 'Hard', '1': 'Relax', '2': 'Left', '3': 'Right'}
            display_labels = [label_mapping.get(l, l) for l in unique_labels]

            for name, proto in models.items():
                with mlflow.start_run(run_name=f"{name}_DS{dataset_id}", nested=True):
                    print(f"Оптимизация Optuna: {name}")

                    def objective(trial):
                        model = clone(proto)
                        p = {}
                        
                        current_model_settings = optuna_settings.get(name, {})
                        
                        for param_name, config in current_model_settings.items():
                            type_ = config[0]
                            
                            if type_ == "int":
                                p[param_name] = trial.suggest_int(param_name, config[1], config[2])
                            elif type_ == "float":
                                p[param_name] = trial.suggest_float(param_name, config[1], config[2])
                            elif type_ == "categorical":
                                p[param_name] = trial.suggest_categorical(param_name, config[1])
                                
                        model.set_params(**p)
                        return cross_val_score(model, X, y_encoded, cv=cv, scoring='accuracy', n_jobs=-1).mean()

                    study = optuna.create_study(direction="maximize")
                    study.optimize(objective, n_trials=15)
                    
                    best_model = clone(proto)
                    best_model.set_params(**study.best_params)
                    
                    start_time = time.time()
                    y_pred = cross_val_predict(best_model, X, y_encoded, cv=cv)
                    best_model.fit(X, y_encoded)
                    duration = time.time() - start_time
                    
                    acc = metrics.accuracy_score(y_encoded, y_pred)
                    f1 = metrics.f1_score(y_encoded, y_pred, average='macro', zero_division=0)

                    mlflow.log_param("model_name", name)
                    mlflow.log_param("dataset_id", dataset_id)
                    mlflow.log_params(study.best_params)
                    mlflow.log_metric("accuracy", acc)
                    mlflow.log_metric("f1_macro", f1)
                    mlflow.log_metric("training_duration", duration)

                    fig, ax = plt.subplots(figsize=(6, 4))
                    sns.heatmap(metrics.confusion_matrix(y_encoded, y_pred), annot=True, fmt='d', cmap='Blues', 
                                xticklabels=display_labels, yticklabels=display_labels)
                    plt.title(f"CM: {name} | Best Acc: {acc:.3f}")
                    
                    plot_filename = f"cm_{name}_ds{dataset_id}.png"
                    full_plot_path = os.path.join(base_artifact_path, plot_filename)
                    fig.savefig(full_plot_path)
                    mlflow.log_artifact(full_plot_path)
                    plt.close(fig)

                    dataset_res.append({
                        'Dataset': dataset_id, 'Model': name, 'Accuracy': acc,
                        'F1_Macro': f1, 'Time': round(duration, 2), 'Params': str(study.best_params)
                    })
                    
    return pd.DataFrame(dataset_res)

## Пайплайны моделей
Использование sklearn.Pipeline гарантирует строгое разделение этапов извлечения признаков и классификации.

In [12]:
models = {
    "Hybrid": Pipeline([
        ('extractor', EEGHybridExtractor(sfreq=sfreq)),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    "Rocket": Pipeline([
        ('convertor', SktimeConverter()),
        ('rocket_classifier', RocketClassifier(random_state=42))
    ]),
    "CSP_LDA": Pipeline([
        ('csp', CSP(log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis())
    ]),
    "CSP_Forest": Pipeline([
        ('csp', CSP(log=True, norm_trace=False)),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])
}

In [13]:
optuna_settings = {
    "Hybrid": {
        'extractor__num_kernels': ("int", 50, 400),
        'classifier__n_estimators': ("int", 100, 500),
        'classifier__max_depth': ("int", 5, 30)
    },
    "Rocket": {
        'rocket_classifier__num_kernels': ("int", 500, 5000)
    },
    "CSP_LDA": {
        'csp__n_components': ("int", 4, 12),
        'csp__reg': ("categorical", ['ledoit_wolf', 'oas', None])
    },
    "CSP_Forest": {
        'csp__n_components': ("int", 4, 12),
        'csp__reg': ("categorical", ['ledoit_wolf', 'oas', None]),
        'classifier__n_estimators': ("int", 100, 500),
        'classifier__max_depth': ("int", 5, 30)
    }
}

# ЗАПУСК ОБУЧЕНИЯ

In [14]:
res = start_training([1, 2, 3], [], StratifiedKFold(n_splits=5, shuffle=True, random_state=42))


--- Обработка Датасета 1 ---
Загрузка EEG BCI
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R07.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R11.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S002\S002R03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
E

[I 2026-03-14 16:23:19,683] A new study created in memory with name: no-name-6c23fc21-5689-46ad-9d1f-7dd2206c5259


Всего эпох после фильтрации: 112
Классы: {np.int64(1): np.int64(54), np.int64(2): np.int64(33), np.int64(3): np.int64(25)}
Оптимизация Optuna: Hybrid


[I 2026-03-14 16:23:26,422] Trial 0 finished with value: 0.5525691699604743 and parameters: {'extractor__num_kernels': 239, 'classifier__n_estimators': 335, 'classifier__max_depth': 11}. Best is trial 0 with value: 0.5525691699604743.
[I 2026-03-14 16:23:31,418] Trial 1 finished with value: 0.5802371541501976 and parameters: {'extractor__num_kernels': 113, 'classifier__n_estimators': 252, 'classifier__max_depth': 7}. Best is trial 1 with value: 0.5802371541501976.
[I 2026-03-14 16:23:33,645] Trial 2 finished with value: 0.5893280632411066 and parameters: {'extractor__num_kernels': 120, 'classifier__n_estimators': 340, 'classifier__max_depth': 27}. Best is trial 2 with value: 0.5893280632411066.
[I 2026-03-14 16:23:36,188] Trial 3 finished with value: 0.5984189723320157 and parameters: {'extractor__num_kernels': 199, 'classifier__n_estimators': 361, 'classifier__max_depth': 7}. Best is trial 3 with value: 0.5984189723320157.
[I 2026-03-14 16:23:38,604] Trial 4 finished with value: 0.632

Оптимизация Optuna: Rocket


...

Computing rank from data with rank=None
    Using tolerance 11 (2.2e-16 eps * 30 dim * 1.7e+15  max singular value)
    Estimated rank (data): 26
    data: rank 26 computed from 30 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 30 -> 26
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 12 (2.2e-16 eps * 30 dim * 1.8e+15  max singular value)
    Estimated rank (data): 26
    data: rank 26 computed from 30 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 30 -> 26
Estimating class=0 covariance using OAS
Done.
Estimating class=1 covariance using OAS
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 11 (2.2e-16 eps * 30 dim * 1.7e+15  

[I 2026-03-14 16:36:57,563] A new study created in memory with name: no-name-8edea200-aa79-40ca-9dab-3c6f5532dc75


Оптимизация Optuna: CSP_Forest


[I 2026-03-14 16:36:58,054] Trial 0 finished with value: 0.5714285714285714 and parameters: {'csp__n_components': 8, 'csp__reg': None, 'classifier__n_estimators': 226, 'classifier__max_depth': 23}. Best is trial 0 with value: 0.5714285714285714.
[I 2026-03-14 16:36:58,979] Trial 1 finished with value: 0.6285714285714286 and parameters: {'csp__n_components': 8, 'csp__reg': 'ledoit_wolf', 'classifier__n_estimators': 491, 'classifier__max_depth': 14}. Best is trial 1 with value: 0.6285714285714286.
[I 2026-03-14 16:36:59,735] Trial 2 finished with value: 0.38571428571428573 and parameters: {'csp__n_components': 6, 'csp__reg': None, 'classifier__n_estimators': 389, 'classifier__max_depth': 22}. Best is trial 1 with value: 0.6285714285714286.
[I 2026-03-14 16:37:00,692] Trial 3 finished with value: 0.5476190476190477 and parameters: {'csp__n_components': 6, 'csp__reg': 'ledoit_wolf', 'classifier__n_estimators': 489, 'classifier__max_depth': 26}. Best is trial 1 with value: 0.628571428571428

Computing rank from data with rank=None
    Using tolerance 11 (2.2e-16 eps * 30 dim * 1.7e+15  max singular value)
    Estimated rank (data): 26
    data: rank 26 computed from 30 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 30 -> 26
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 12 (2.2e-16 eps * 30 dim * 1.8e+15  max singular value)
    Estimated rank (data): 26
    data: rank 26 computed from 30 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 30 -> 26
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 11 (

### Вывод таблицы результатов

In [15]:
for dataset_id in sorted(res['Dataset'].unique()):
    print("Датасет :", dataset_id)
    dataset_df = res[res['Dataset'] == dataset_id].copy()
    
    dataset_df = dataset_df.set_index('Model').drop(columns=['Dataset'])
    display(dataset_df.style.set_properties(**{'text-align': 'left'}))

Датасет : 1


,Accuracy,F1_Macro,Time,Params
Model,,,,
Hybrid,0.642857,0.555676,9.220000,"{'extractor__num_kernels': 212, 'classifier__n_estimators': 268, 'classifier__max_depth': 20}"
Rocket,0.767857,0.744544,53.320000,{'rocket_classifier__num_kernels': 1562}
CSP_LDA,0.482143,0.411612,13.920000,"{'csp__n_components': 10, 'csp__reg': None}"
CSP_Forest,0.517857,0.423180,15.370000,"{'csp__n_components': 6, 'csp__reg': None, 'classifier__n_estimators': 232, 'classifier__max_depth': 13}"


Датасет : 2


,Accuracy,F1_Macro,Time,Params
Model,,,,
Hybrid,0.581818,0.567521,4.970000,"{'extractor__num_kernels': 264, 'classifier__n_estimators': 215, 'classifier__max_depth': 18}"
Rocket,0.563636,0.560000,21.120000,{'rocket_classifier__num_kernels': 646}
CSP_LDA,0.527273,0.527116,0.490000,"{'csp__n_components': 4, 'csp__reg': None}"
CSP_Forest,0.636364,0.633333,3.260000,"{'csp__n_components': 4, 'csp__reg': 'ledoit_wolf', 'classifier__n_estimators': 483, 'classifier__max_depth': 10}"


Датасет : 3


,Accuracy,F1_Macro,Time,Params
Model,,,,
Hybrid,0.666667,0.655271,2.280000,"{'extractor__num_kernels': 360, 'classifier__n_estimators': 103, 'classifier__max_depth': 6}"
Rocket,0.696970,0.694444,8.080000,{'rocket_classifier__num_kernels': 541}
CSP_LDA,0.575758,0.572222,0.250000,"{'csp__n_components': 4, 'csp__reg': 'oas'}"
CSP_Forest,0.666667,0.665438,2.020000,"{'csp__n_components': 7, 'csp__reg': 'ledoit_wolf', 'classifier__n_estimators': 312, 'classifier__max_depth': 6}"
